# 01 · Run one task — training, then everything that needs this model

**Set `TASK` in the first cell and launch. This notebook is the unit of concurrency.** Run as
many copies at once as your Kaggle quota allows: each one writes only its own output, reads
nothing another worker writes, and needs no coordination. Nothing here has to be run in order.

`python scripts/training/tasks.py` lists all 25 task names. To launch several at once, fork
this notebook once per task and change the one line.

| Tier | Tasks |
| --- | --- |
| A | `cnn-fold0` … `cnn-fold4`, `tf-fold0` … `tf-fold4` |
| B | `cnn-loso_<Source>`, `tf-loso_<Source>`, `identity-fold0` … `identity-fold4`, `<arm>-fold<r>-seed1/2` |

## What one session does, and why it stays inside both limits

1. trains, stopping cleanly ~1.5 h before the session cap with a checkpoint on disk
2. if the run finished, writes **its own** out-of-fold predictions
3. occludes and re-infers **only the cases this fold held out** — the correct pairing (the
   model never saw them) and the only version with a bounded disk cost
4. predicts the NIH negatives when this task is the one that should
5. prunes every checkpoint the study does not evaluate, and asserts the output budget

**Time.** A pre-registered run is ~250,000 steps and spans many sessions. The trainer stops on
a wall-clock budget and raises rather than returning, because returning normally would make
nnU-Net write `checkpoint_final.pth` and mark an unfinished run as finished. Re-launch with
the same `TASK` and it resumes from `checkpoint_latest.pth`.

`RESERVE_HOURS` is held back from the training budget and has to cover three things, only one
of which this notebook controls: nnU-Net's own end-of-training validation pass over the
held-out fold (which happens inside training, after the last epoch), the occlusion inference,
and saving. Two hours is a starting estimate, not a measurement — **the first worker session
tells you the real numbers**, and the sensible response to running short is not a bigger
reserve but a second session: re-launching a finished task skips training entirely and does
inference only, resuming any prediction directory that was left partial.

**Storage.** One session holds one checkpoint plus a few hundred MB of predictions — about
2 GB against a 20 GB cap. The cap is per output, so twenty-five runs are twenty-five 2 GB
outputs, not one 50 GB pile. Preprocessed data never touches the output: it comes from an
attached dataset, or is regenerated into scratch.

In [ ]:
# ==========================================================================================
TASK = "cnn-fold0"     # one of `python scripts/training/tasks.py`
EPOCHS = None          # None = the frozen 1000-epoch schedule. A number is a DEVIATION.
RESERVE_HOURS = 2.0    # see the note below on what this has to cover
# ==========================================================================================

In [ ]:
# === PDAC study bootstrap =============================================================
# Identical in all three notebooks. Paths, environment, repo, disk budget, and the
# cross-session state helpers.
import os, sys, subprocess, shutil, json, tarfile, textwrap, time
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
REPO_URL  = "https://github.com/spraldev/pdac-inductive-bias.git"

# --- the two hard limits, in one place -------------------------------------------------
# Kaggle kills a session at 12 h and refuses to save an output larger than 20 GB. Both are
# silent failures if you meet them by accident, so both are budgeted with headroom and
# checked rather than hoped for.
SESSION_HOURS   = float(os.environ.get("PDAC_SESSION_HOURS", 11.0))   # of a 12 h cap
OUTPUT_LIMIT_GB = float(os.environ.get("PDAC_OUTPUT_LIMIT_GB", 17.0)) # of a 20 GB cap
SESSION_T0 = time.time()

# --- paths ------------------------------------------------------------------------------
# WORK persists as the notebook's saved output and is what the 20 GB cap applies to: only
# small, precious, or genuinely needed-downstream things go there. SCRATCH is much larger and
# is wiped with the session, so everything regenerable lives there — raw images, nnU-Net
# preprocessed data, occluded volumes.
if ON_KAGGLE:
    WORK    = Path("/kaggle/working")
    SCRATCH = Path("/kaggle/temp/pdac"); SCRATCH.mkdir(parents=True, exist_ok=True)
    REPO    = WORK / "pdac-research"
else:
    WORK    = Path(os.environ.get("PDAC_WORK", Path.cwd() / "pdac_work"))
    SCRATCH = Path(os.environ.get("PDAC_SCRATCH", WORK / "scratch"))
    REPO    = Path(os.environ.get("PDAC_REPO", Path.cwd()))
    WORK.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(os.environ.get("PDAC_DATA", SCRATCH / "data"))
RESULTS   = WORK / "results";  RESULTS.mkdir(parents=True, exist_ok=True)
STATE     = WORK / "state";    STATE.mkdir(parents=True, exist_ok=True)
PREDS     = WORK / "preds";    PREDS.mkdir(parents=True, exist_ok=True)

# --- repo ---------------------------------------------------------------------------------
def _find_attached(*required):
    """First attached input containing all of the given relative paths."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for p in sorted(root.glob("*")):
        for cand in [p] + sorted(x for x in p.glob("*") if x.is_dir()):
            if all((cand / r).exists() for r in required):
                return cand
    return None

if ON_KAGGLE and not (REPO / "config" / "analysis_config.yaml").exists():
    src = _find_attached("config/analysis_config.yaml")
    if src is not None:
        print(f"Using repo attached as a dataset: {src}")
        shutil.copytree(src, REPO, dirs_exist_ok=True)
    else:
        print("Cloning the repo (Internet must be on in notebook settings) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts" / "analysis"))
sys.path.insert(0, str(REPO / "scripts" / "training"))
os.environ["PDAC_REPO"] = str(REPO)

# These notebooks are generated FROM the repo but, on Kaggle, run AGAINST a clone of it — so
# an uncommitted or unpushed change is invisible here no matter how current the notebook is.
# When the clone predates the notebook the symptom lands far from the cause: an old test
# asserting old counts, a script missing a flag this notebook passes. Checking the contract
# up front turns that into one clear message.
_REQUIRED = ["config/analysis_config.yaml", "scripts/training/tasks.py",
             "scripts/analysis/build_per_case_table.py", "scripts/analysis/make_figures.py",
             "src/trainers/budget_trainers.py", "scripts/kaggle/pack_for_kaggle.py"]
_missing = [r for r in _REQUIRED if not (REPO / r).exists()]
if _missing:
    raise RuntimeError(
        "The repo this notebook is running against is older than the notebook itself.\n"
        f"  missing: {_missing}\n"
        f"  repo:    {REPO}\n"
        "These notebooks clone " + REPO_URL + ", so local commits only reach Kaggle "
        "once they are PUSHED. "
        "Either push, or upload the repo as a Kaggle Dataset and attach it "
        "(the bootstrap prefers an attached input containing config/analysis_config.yaml).")

# --- nnU-Net environment --------------------------------------------------------------------
# raw and preprocessed are regenerable and enormous -> SCRATCH.
# results holds checkpoints, which are neither -> WORK, under the budget guard below.
os.environ["nnUNet_raw"]          = str(SCRATCH / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(SCRATCH / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(WORK / "nnUNet_results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

# --- shell / install helpers -------------------------------------------------------------------
def sh(cmd, cwd=None, check=True):
    """Run a shell command from the repo root, streaming output into the notebook."""
    cwd = str(cwd or REPO)
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

def pip_install(pkgs, quiet=True):
    sh(f"{sys.executable} -m pip install {'-q ' if quiet else ''}--no-warn-script-location {pkgs}")

def gpu_info():
    try:
        import torch
    except ImportError:
        print("torch not installed yet"); return None
    if not torch.cuda.is_available():
        print("No CUDA device. Turn on a GPU accelerator in notebook settings."); return None
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({gb:.1f} GB)")
    return {"name": name, "vram_gb": round(gb, 1)}

# --- the 20 GB guard ------------------------------------------------------------------------------
def dir_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e9

def disk_report(detail=True):
    """What the session is holding, split by which limit it counts against."""
    out = dir_gb(WORK)
    free_scratch = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"OUTPUT (counts against the {OUTPUT_LIMIT_GB:.0f}/20 GB cap): {out:.2f} GB")
    if detail:
        for sub in sorted(p for p in WORK.iterdir() if p.is_dir()):
            g = dir_gb(sub)
            if g > 0.01:
                print(f"    {g:7.2f} GB  {sub.name}/")
    print(f"SCRATCH (wiped with the session, not capped): {dir_gb(SCRATCH):.2f} GB used, "
          f"{free_scratch:.0f} GB free")
    return out

def enforce_output_budget(limit_gb=None, where=""):
    """Get the output back under budget, and only then complain if it cannot be done.

    Raising alone would not help: Kaggle refuses the save regardless of what the notebook
    thinks, so an over-budget session loses its GPU hours either way. This frees space in
    increasing order of regret and re-measures after each step, so the common case (a
    checkpoint the study never evaluates) is handled silently and only a genuine overrun
    reaches the user.
    """
    limit = OUTPUT_LIMIT_GB if limit_gb is None else limit_gb
    used = dir_gb(WORK)
    if used <= limit:
        print(f"output {used:.2f} / {limit:.0f} GB{' at ' + where if where else ''}  OK")
        return used

    print(f"output {used:.2f} GB is over the {limit:.0f} GB budget — freeing space")

    # 1. Checkpoints this study never reads. Zero regret.
    prune_checkpoints()
    used = dir_gb(WORK)

    # 2. Softmax dumps and validation scratch. Nothing here reads them either; they only
    #    appear if a training command was run with --npz, which this repo no longer does.
    if used > limit:
        for pattern in ("*.npz", "*.pkl"):
            for f in Path(os.environ["nnUNet_results"]).rglob(pattern):
                print(f"  removed {f.name}"); f.unlink()
        used = dir_gb(WORK)

    # 3. Resume checkpoints for runs that finished. Costs the ability to resume a run that
    #    has nothing left to resume.
    if used > limit:
        prune_checkpoints(keep_latest_if_unfinished=False)
        used = dir_gb(WORK)

    if used > limit:
        disk_report()
        raise RuntimeError(
            f"Output is still {used:.1f} GB after pruning, over the {limit:.0f} GB budget"
            f"{' at ' + where if where else ''}. Kaggle will refuse to save this session. "
            "Drop this task's checkpoint (DROP_CHECKPOINT = True) if its predictions are "
            "already written — every analysis except the receptive-field measurement reads "
            "predictions, not weights.")
    print(f"output now {used:.2f} / {limit:.0f} GB  OK")
    return used

def time_left_h():
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600.0

def check_time(where=""):
    left = time_left_h()
    print(f"{left:.2f} h left of the {SESSION_HOURS:.1f} h session budget"
          f"{' at ' + where if where else ''}")
    return left

def prune_checkpoints(root=None, keep_latest_if_unfinished=True):
    """Keep exactly what the study needs from each run's directory.

    nnU-Net writes checkpoint_best, checkpoint_latest and checkpoint_final. Evaluation in this
    study is on checkpoint_final only (the pre-registered schedule has no early stopping), so
    best is always removable, and latest is removable the moment final exists. Left alone,
    three checkpoints per run is three times the storage for no gain.
    """
    root = Path(root or os.environ["nnUNet_results"])
    freed = 0.0
    for fold_dir in sorted(p for p in root.rglob("fold_*") if p.is_dir()):
        final = fold_dir / "checkpoint_final.pth"
        drop = [fold_dir / "checkpoint_best.pth"]
        if final.exists() or not keep_latest_if_unfinished:
            drop.append(fold_dir / "checkpoint_latest.pth")
        for f in drop:
            if f.exists():
                freed += f.stat().st_size / 1e9
                f.unlink()
                print(f"  removed {f.relative_to(root)}")
    if freed:
        print(f"freed {freed:.2f} GB")
    return freed

# --- cross-session state --------------------------------------------------------------------------
def save_state(*rel_paths):
    """Copy repo-relative paths into WORK/state so they survive as notebook output."""
    for rel in rel_paths:
        src = REPO / rel
        if not src.exists():
            print(f"  (skip, absent) {rel}"); continue
        dst = STATE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        (shutil.copytree if src.is_dir() else shutil.copy2)(
            src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
        print(f"  saved {rel}")

def restore_state(*rel_paths, required=True):
    """Restore from WORK/state or from any attached dataset holding a state/ directory."""
    sources = [STATE]
    if Path("/kaggle/input").exists():
        sources += sorted(Path("/kaggle/input").rglob("state"))
    missing = []
    for rel in rel_paths:
        for base in sources:
            src = base / rel
            if src.exists():
                dst = REPO / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                (shutil.copytree if src.is_dir() else shutil.copy2)(
                    src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
                print(f"  restored {rel}  <- {base}")
                break
        else:
            missing.append(rel)
    if missing:
        msg = ("Missing state: " + ", ".join(missing) + "\n  Run the preparation notebook, "
               "then attach its output here (Add Input -> Your Work).")
        if required:
            raise FileNotFoundError(msg)
        print("  " + msg)
    return not missing

def restore_chunks(dest, manifest_name="pack_manifest.json"):
    """Unpack a chunked dataset produced by scripts/kaggle/pack_for_kaggle.py.

    Large reusable data (the preprocessed cohort) cannot travel as notebook output — that is
    what the 20 GB cap forbids — so it travels as an attached dataset in size-bounded parts.
    This finds the manifest in any attached input and extracts every part into dest.
    """
    root = Path("/kaggle/input")
    if not root.exists():
        return False
    for man_path in sorted(root.rglob(manifest_name)):
        man = json.loads(man_path.read_text())
        parts = man.get("parts", [])
        print(f"Found a {len(parts)}-part pack at {man_path.parent} "
              f"({man.get('uncompressed_bytes', 0)/1e9:.1f} GB)")
        dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
        for entry in parts:
            src = man_path.parent / entry["name"]
            if not src.exists():
                print(f"  MISSING {entry['name']} — attach every part, not just some"); continue
            with tarfile.open(src) as tar:
                tar.extractall(dest)
            print(f"  extracted {entry['name']} ({entry['n_files']} files)")
        return True
    return False

print(f"ON_KAGGLE={ON_KAGGLE}\nREPO={REPO}\nWORK={WORK}\nSCRATCH={SCRATCH}\n"
      f"DATA_ROOT={DATA_ROOT}\nbudgets: {SESSION_HOURS} h session, {OUTPUT_LIMIT_GB} GB output")

In [ ]:
# Analysis environment. Kaggle already ships numpy/pandas/scipy/matplotlib; these are the rest.
pip_install("SimpleITK nibabel openpyxl pyyaml statsmodels zenodo-get "
            "'surface-distance @ git+https://github.com/google-deepmind/surface-distance.git'")
import importlib
for m in ("SimpleITK", "surface_distance", "statsmodels", "yaml", "pandas", "scipy"):
    importlib.import_module(m)
print("analysis environment OK")

In [ ]:
restore_state("splits", "config/frozen_thresholds.yaml", "preregistration/DEVIATIONS.md")
import yaml, pandas as pd
frozen = yaml.safe_load(open(REPO / "config" / "frozen_thresholds.yaml"))
arch = frozen.get("architecture")

# The transformer trainer name is needed before nnU-Net is installed (the trainer shim imports
# it), so fall back to the config's candidate list until the first worker records the real one.
cfg = yaml.safe_load(open(REPO / "config" / "analysis_config.yaml"))
PRIMUS_TRAINER = (arch or {}).get("transformer", {}).get("trainer")
CNN_PLANS = (arch or {}).get("cnn", {}).get("plans")
if not PRIMUS_TRAINER:
    PRIMUS_TRAINER = "nnUNet_PrimusV2S_Trainer"
    CNN_PLANS = "nnUNetResEncUNetMPlans"
    print("No architecture record yet — using the provisional pairing and measuring it below. "
          "The first worker to get here freezes the measured numbers.")
os.environ["PRIMUS_BASE_TRAINER"] = PRIMUS_TRAINER

os.environ["PDAC_SAVE_EVERY"] = "5"
if EPOCHS:
    os.environ["PDAC_EPOCHS"] = str(int(EPOCHS))
BUDGET_SUFFIX = "_budget" + (f"{int(EPOCHS)}ep" if EPOCHS else "")
print(f"TASK={TASK}  CNN={CNN_PLANS}  transformer={PRIMUS_TRAINER}  suffix={BUDGET_SUFFIX}")
gpu = gpu_info()

In [ ]:
# nnU-Net from master: the PrimusV2 trainers are not guaranteed to be in the PyPI release.
# The commit is pinned into the frozen config the first time this runs, so every later session
# and every collaborator gets the same one.
import yaml
frozen_path = REPO / "config" / "frozen_thresholds.yaml"
frozen = (yaml.safe_load(open(frozen_path)) or {}) if frozen_path.exists() else {}
pinned = (frozen.get("architecture") or {}).get("nnunet_commit")
pinned = pinned.split("@")[-1].strip() if pinned and "@" in str(pinned) else None

spec = "git+https://github.com/MIC-DKFZ/nnUNet.git" + (f"@{pinned}" if pinned else "")
print(f"Installing nnunetv2 from {spec}")
pip_install(f"'nnunetv2 @ {spec}'")
import nnunetv2
sh("pip freeze | grep -i nnunet")

In [ ]:
# Put this study's custom trainers where nnU-Net's class finder looks. A shim module rather
# than a copy, so each trainer's own __file__ still points into the repo — budget_trainers.py
# and seed_variant_trainers.py both read the frozen config relative to it.
import nnunetv2
variants = Path(nnunetv2.__path__[0]) / "training" / "nnUNetTrainer" / "variants"
(variants / "pdac_custom_trainers.py").write_text(textwrap.dedent(f"""
    import sys
    if {str(REPO)!r} not in sys.path:
        sys.path.insert(0, {str(REPO)!r})
    from src.trainers.primus_identity_trainer import nnUNet_PrimusV2_Identity_Trainer  # noqa: F401
    from src.trainers.seed_variant_trainers import *  # noqa: F401,F403
    from src.trainers.budget_trainers import *  # noqa: F401,F403
"""))

from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
search = str(Path(nnunetv2.__path__[0]) / "training" / "nnUNetTrainer")
def trainer_exists(name):
    return recursive_find_python_class(
        search, name, current_module="nnunetv2.training.nnUNetTrainer") is not None
for name in (PRIMUS_TRAINER, "nnUNet_PrimusV2_Identity_Trainer",
             "nnUNetTrainer" + BUDGET_SUFFIX, PRIMUS_TRAINER + BUDGET_SUFFIX):
    print(f"  {'OK  ' if trainer_exists(name) else 'MISS'} {name}")

In [ ]:
# --- resolve the task ------------------------------------------------------------------------
# One place maps a task name to its trainer, plans, nnU-Net fold index, and results directory,
# so the run log, the checkpoint location, and the predictions can never disagree.
sys.path.insert(0, str(REPO / "scripts" / "training"))
import importlib, tasks as task_mod
importlib.reload(task_mod)
T = task_mod.parse(TASK, repo=REPO, budget_suffix=BUDGET_SUFFIX)
print(json.dumps(T, indent=1))

if EPOCHS:
    print("=" * 78)
    print(f"DEVIATION: {EPOCHS} epochs instead of the frozen {cfg['training']['epochs']}.")
    print("This run writes to its own results directory and is not a study result until the")
    print("deviation is recorded in preregistration/DEVIATIONS.md.")
    print("=" * 78)
    import datetime
    dev = REPO / "preregistration" / "DEVIATIONS.md"
    dev.write_text(dev.read_text().replace(
        "| — | — | none to date | — | — |",
        f"| {datetime.date.today().isoformat()} | training.epochs | "
        f"{cfg['training']['epochs']} -> {EPOCHS} | compute budget | (not yet tagged) |"
        + chr(10) + "| — | — | none to date | — | — |"))

In [ ]:
# --- preprocessed data: attached dataset, or regenerate into scratch ---------------------------
# Never into the output: it is large and exactly regenerable, and the 20 GB cap is for things
# that are neither.
ds_dir = Path(os.environ["nnUNet_preprocessed"]) / "Dataset501_PDAC"
if not (ds_dir / "dataset.json").exists():
    if restore_chunks(Path(os.environ["nnUNet_preprocessed"])):
        print("preprocessed data restored from an attached pack — preprocessing skipped")
    else:
        # Preprocessing is not interruptible and not time-checkable once started, so the
        # check is here. Attaching a preprocessed pack removes this risk entirely and is the
        # single biggest thing you can do for the session budget.
        if time_left_h() < 4:
            raise RuntimeError(
                f"Only {time_left_h():.1f} h left and preprocessing has not started. It is not "
                "interruptible, so starting now risks spending the whole session on it and "
                "training for nothing. Attach a preprocessed pack (see "
                "scripts/kaggle/pack_for_kaggle.py) or start a fresh session.")
        print("No preprocessed pack attached; regenerating (expected on a fresh session).")
        sh(f"{sys.executable} scripts/data/convert_to_nnunet.py --data-root {DATA_ROOT} "
           f"--cohort splits/cohort.csv")
        planner = ("nnUNetPlannerResEnc"
                   + CNN_PLANS.replace("nnUNetResEncUNet", "").replace("Plans", ""))
        sh(f"nnUNetv2_plan_and_preprocess -d 501 -pl {planner}")

src = REPO / "splits" / ("splits_with_loso.json"
                         if (REPO / "splits" / "splits_with_loso.json").exists()
                         else "splits_final.json")
shutil.copy(src, ds_dir / "splits_final.json")
print(f"{len(json.load(open(ds_dir / 'splits_final.json')))} folds installed from {src.name}")
check_time("before training")

In [ ]:
# --- record the matched-budget architecture, if this is the first worker to get here -----------
# Both arms measured on the same card at the same patch size. record_arch_stats.py writes it
# once and refuses to run again, so whichever worker arrives first freezes it and the rest
# read it back.
if not arch:
    import torch, time as _time
    from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
    from nnunetv2.utilities.get_network_from_plans import get_network_from_plans
    from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
    import nnunetv2

    plans = json.load(open(ds_dir / f"{CNN_PLANS}.json"))
    patch_xyz = plans["configurations"]["3d_fullres"]["patch_size"]
    dj = json.load(open(ds_dir / "dataset.json"))
    n_in, n_out = len(dj["channel_names"]), len(dj["labels"])

    def measure(build, label, batch=2, steps=3):
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        net = build().cuda().train()
        n_params = sum(p.numel() for p in net.parameters())
        opt = torch.optim.SGD(net.parameters(), lr=1e-2)
        x = torch.randn(batch, n_in, *patch_xyz[::-1], device="cuda")
        y = torch.randint(0, n_out, (batch, *patch_xyz[::-1]), device="cuda")
        t0 = None
        for i in range(steps + 1):
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = net(x)
                out = out[0] if isinstance(out, (list, tuple)) else out
                loss = torch.nn.functional.cross_entropy(out.float(), y)
            loss.backward(); opt.step(); torch.cuda.synchronize()
            if i == 0:
                t0 = _time.time()     # first step includes allocator warm-up; not timed
        step_s = (_time.time() - t0) / steps
        peak = torch.cuda.max_memory_allocated() / 1e9
        del net, opt, x, y; torch.cuda.empty_cache()
        print(f"{label}: {n_params/1e6:.1f}M params, peak {peak:.1f} GB, {step_s:.2f} s/step")
        return n_params, peak, step_s

    pm = PlansManager(plans); cm = pm.get_configuration("3d_fullres")
    cnn_p, cnn_v, cnn_s = measure(
        lambda: get_network_from_plans(cm.network_arch_class_name, cm.network_arch_init_kwargs,
                                       cm.network_arch_init_kwargs_req_import, n_in, n_out,
                                       allow_init=True, deep_supervision=False), "CNN   ")
    search = str(Path(nnunetv2.__path__[0]) / "training" / "nnUNetTrainer")
    PrimusCls = recursive_find_python_class(search, PRIMUS_TRAINER,
                                            current_module="nnunetv2.training.nnUNetTrainer")
    try:
        tf_p, tf_v, tf_s = measure(
            lambda: PrimusCls.build_network_architecture(
                cm.network_arch_class_name, cm.network_arch_init_kwargs,
                cm.network_arch_init_kwargs_req_import, n_in, n_out, False), "Primus")
    except Exception as e:
        raise SystemExit(
            f"Could not build {PRIMUS_TRAINER}'s network directly ({type(e).__name__}: {e}). "
            "Measure the transformer arm from a real 1-epoch run and pass the numbers to "
            "record_arch_stats.py by hand. Do not record estimates: the matched-budget "
            "ceiling is the study's only control, and an unmeasured ceiling is not one.")

    ratio = max(cnn_v, tf_v) / max(min(cnn_v, tf_v), 1e-9)
    print(f"VRAM ratio between arms: {ratio:.2f}x")
    if ratio > 1.25:
        print("*** The arms are NOT on a matched VRAM budget. That breaks the study's only "
              "control and is a same-day escalation. Fix the pairing before training. ***")
    sh(f"{sys.executable} scripts/training/record_arch_stats.py "
       f"--patch-size {patch_xyz[2]} {patch_xyz[1]} {patch_xyz[0]} "
       f'--nnunet-commit "$(pip freeze | grep -i nnunetv2)" --gpu "{(gpu or {}).get("name","unknown")}" '
       f"--cnn-plans {CNN_PLANS} --cnn-params {cnn_p} --cnn-vram-gb {cnn_v:.2f} "
       f"--cnn-step-time-s {cnn_s:.3f} --primus-trainer {PRIMUS_TRAINER} "
       f"--tf-params {tf_p} --tf-vram-gb {tf_v:.2f} --tf-step-time-s {tf_s:.3f}", check=False)
    save_state("config/frozen_thresholds.yaml")
else:
    print("architecture already frozen:")
    print(yaml.safe_dump(arch, sort_keys=False))

In [ ]:
# --- restore this task's checkpoint from an earlier session -------------------------------------
# Attach the previous session of THIS task (Add Input -> Your Work). Only this task's
# directory is copied: pulling in other workers' checkpoints would spend the output budget on
# models this session has no use for.
RES = Path(os.environ["nnUNet_results"])
out_dir = RES / "Dataset501_PDAC" / T["results_subdir"] / f"fold_{T['nnunet_fold']}"
restored = 0
if Path("/kaggle/input").exists():
    for cand in Path("/kaggle/input").rglob(f"Dataset501_PDAC/{T['results_subdir']}"):
        for f in cand.rglob("*"):
            if f.is_file() and f.suffix in (".pth", ".json", ".txt", ".pkl"):
                dst = RES / "Dataset501_PDAC" / T["results_subdir"] / f.relative_to(cand)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists():
                    shutil.copy2(f, dst); restored += 1
print(f"restored {restored} file(s) for {T['results_subdir']}")

final = out_dir / "checkpoint_final.pth"
latest = out_dir / "checkpoint_latest.pth"
print("FINISHED" if final.exists() else ("resuming from checkpoint_latest.pth"
                                         if latest.exists() else "fresh start"))
disk_report()

In [ ]:
# --- train ---------------------------------------------------------------------------------------
# The wall-clock budget is whatever is left minus the reserve, so a session that spent two
# hours preprocessing trains for correspondingly less and still saves cleanly.
if final.exists():
    print("Already finished — skipping training.")
else:
    budget = max(0.25, time_left_h() - RESERVE_HOURS)
    os.environ["PDAC_MAX_HOURS"] = f"{budget:.3f}"
    print(f"training budget for this session: {budget:.2f} h")

    commit = subprocess.run("pip freeze | grep -i nnunetv2", shell=True, capture_output=True,
                            text=True).stdout.strip() or "unknown"
    run_id = subprocess.run(
        [sys.executable, "scripts/training/log_run.py", "start", "--arm", T["arm"],
         "--fold", T["fold_label"], "--seed", T["seed"], "--nnunet-commit", commit,
         "--trainer-or-plans", T["trainer"] + (f" / {T['plans']}" if T["plans"] else ""),
         "--gpu", (gpu or {}).get("name", "unknown"),
         "--notes", f"kaggle worker; task={TASK}; budget={budget:.2f}h"],
        cwd=REPO, capture_output=True, text=True, check=True).stdout.strip()
    print("run_id:", run_id)

    cmd = f"nnUNetv2_train {T['train_cmd_args']}" + (" --c" if latest.exists() else "")
    rc = sh(cmd, check=False)

    if final.exists():
        status, note = "completed", "run finished"
    elif latest.exists():
        status, note = "crashed", "PAUSED on the wall-clock budget; re-launch to resume"
        print("\n>>> Paused, not failed. Save this version, attach it to the next session, "
              "and run the same TASK again.")
    else:
        status, note = "crashed", f"no checkpoint written; exit {rc}"
    sh(f"{sys.executable} scripts/training/log_run.py finish --run-id {run_id} "
       f"--status {status} --checkpoint-path {final} --notes \"{note}; exit {rc}\"")

check_time("after training")
disk_report()

In [ ]:
# --- predictions, but only if the run actually finished -------------------------------------------
# nnU-Net writes this fold's held-out predictions during on_train_end, so out-of-fold
# predictions come out of training itself rather than a second inference pass.
FINISHED = final.exists()
val_dir = out_dir / "validation"
arm_tag = {"cnn": "cnn", "tf": "tf", "identity": "identity_control"}[T["arm_key"]]

if FINISHED and val_dir.exists():
    dst = PREDS / arm_tag / T["pred_key"]
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in val_dir.glob("*.nii.gz"):
        shutil.copy2(f, dst / f.name); n += 1
    print(f"{n} out-of-fold predictions -> {dst}")
else:
    print("Run not finished — no predictions this session. Everything below is skipped.")

In [ ]:
# --- occlusion, for this fold's held-out cases only (H2) --------------------------------------------
# Occluding the whole cohort centrally writes three copies of every image. Doing it per fold
# keeps the disk cost proportional to one fold AND gets the pairing right: every case here was
# held out by this model, so its occlusion Dice loss is measured on a case it never saw.
# Replicates and LOSO runs are excluded: the occlusion test is defined on the two arms'
# cross-validation folds, and running it again per replicate would re-do identical work into a
# path that already has an owner.
DO_OCCLUSION = (FINISHED and T["arm_key"] in ("cnn", "tf")
                and not T["is_replicate"] and not T["fold_label"].startswith("loso_"))
if DO_OCCLUSION and time_left_h() > 0.5:
    OCC = SCRATCH / "occlusion"
    sh(f"{sys.executable} scripts/analysis/occlusion_test.py --stage occlude "
       f"--cohort splits/cohort.csv --strata splits/strata.csv --workdir {OCC} "
       f"--fold {T['nnunet_fold']}")
    for lo, hi in cfg["hypotheses"]["h2"]["occlusion_shells_mm"]:
        if time_left_h() < 0.3:
            print("Out of session time — remaining shells next session."); break
        src_dir = OCC / "occluded" / f"shell_{lo}_{hi}" / "imagesTs"
        dst_dir = PREDS / "occlusion" / arm_tag / f"shell_{lo}_{hi}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        flag, value = ("-p", T["plans"]) if T["plans"] else ("-tr", T["trainer"])
        extra = f" -tr {T['trainer']}" if T["plans"] else ""
        # --continue_prediction: skip cases already written, so a session that ran out of
        # time mid-shell resumes instead of redoing an hour of inference.
        sh(f"nnUNetv2_predict -i {src_dir} -o {dst_dir} -d 501 -c 3d_fullres "
           f"-f {T['nnunet_fold']} {flag} {value}{extra} --continue_prediction", check=False)
        n_done = len(list(dst_dir.glob("*.nii.gz")))
        n_want = len(list(src_dir.glob("*.nii.gz")))
        print(f"shell {lo}-{hi}: {n_done}/{n_want} predictions"
              + ("" if n_done >= n_want else "  (partial — re-launch this task to finish)"))
        enforce_output_budget(where=f"after shell {lo}-{hi}")
else:
    print("Occlusion skipped for this task (it applies to the two arms' CV folds).")

In [ ]:
# --- NIH negative control ------------------------------------------------------------------------------
# The NIH cases have no lesions and are in no training fold, so they need a real inference pass.
# In domain = a CV-fold model; under source shift = a leave-one-source-out model, which never
# saw the acquisition source it is being asked about. Only the tasks that should do this, do it.
cohort = pd.read_csv(REPO / "splits" / "cohort.csv")
nih = cohort[cohort.get("source", pd.Series(dtype=str)) == "NIH"] if "source" in cohort else cohort.iloc[0:0]
CONDITION = (None if T["is_replicate"]
             else "in_domain" if T["fold_label"] == "fold0"
             else "source_shift" if T["fold_label"].startswith("loso_") else None)

if FINISHED and CONDITION and T["arm_key"] in ("cnn", "tf") and len(nih) and time_left_h() > 0.4:
    import SimpleITK as sitk
    nih_in = SCRATCH / "nih_images"; nih_in.mkdir(parents=True, exist_ok=True)
    for r in nih.itertuples():
        dst = nih_in / f"{r.case_id}_0000.nii.gz"
        if dst.exists():
            continue
        if str(r.path).endswith(".nii.gz"):
            shutil.copy2(r.path, dst)
        else:
            sitk.WriteImage(sitk.ReadImage(str(r.path)), str(dst), useCompression=True)
    out = PREDS / "nih" / arm_tag / CONDITION
    out.mkdir(parents=True, exist_ok=True)
    flag, value = ("-p", T["plans"]) if T["plans"] else ("-tr", T["trainer"])
    extra = f" -tr {T['trainer']}" if T["plans"] else ""
    sh(f"nnUNetv2_predict -i {nih_in} -o {out} -d 501 -c 3d_fullres "
       f"-f {T['nnunet_fold']} {flag} {value}{extra} --continue_prediction", check=False)
    print(f"NIH {arm_tag}/{CONDITION}: {len(list(out.glob('*.nii.gz')))} predictions")
else:
    print(f"NIH inference not this task's job (condition={CONDITION}).")

In [ ]:
# --- prune, budget-check, hand off ----------------------------------------------------------------------
# checkpoint_best is never evaluated by this study (the frozen schedule has no early stopping)
# and checkpoint_latest is dead weight once final exists. Left alone that is three checkpoints
# per run for no gain.
prune_checkpoints()

# The ERF measurement needs one real CNN checkpoint; every other analysis reads predictions.
# If the output is tight, dropping this task's checkpoint costs only the ability to re-infer.
DROP_CHECKPOINT = False
if DROP_CHECKPOINT and FINISHED:
    for f in out_dir.glob("checkpoint_*.pth"):
        print("dropping", f.name); f.unlink()

# The run log is per-session; the analysis notebook merges them. Naming it by task means two
# concurrent workers never collide.
log = REPO / "scripts" / "training" / "run_log.csv"
if log.exists():
    (STATE / "run_logs").mkdir(parents=True, exist_ok=True)
    shutil.copy2(log, STATE / "run_logs" / f"{TASK}.csv")
save_state("config/frozen_thresholds.yaml", "preregistration/DEVIATIONS.md")

disk_report()
enforce_output_budget(where="end of worker")
print(f"\nTASK {TASK}: {'FINISHED' if FINISHED else 'PAUSED — re-launch to resume'}")

## Finishing

**Save Version → Save & Run All.**

- **Paused?** Attach this session to the next one and run the same `TASK` again. It resumes
  from `checkpoint_latest.pth`; nothing is lost and nothing is repeated.
- **Finished?** Its predictions are in the output. Move on to another task — or launch several
  at once, since no worker reads another worker's output.

When enough tasks are done, attach all of them to **02_analyze**. Tier A analysis needs the ten
`cnn-fold*` and `tf-fold*` outputs; the Tier B sections light up as their tasks land, and the
analysis notebook reports what is missing rather than silently analysing a partial study.